# Notebook 05 — DiT Staged Fine-Tuning

Trains **`microsoft/dit-base`** with a three-stage strategy:

| Stage | Unfrozen | Learning Rate |
|-------|----------|---------------|
| 1     | Head only | `stage1_lr` |
| 2     | Head + top 2 encoder blocks | `stage2_lr` |
| 3     | Full model | `stage3_lr` |

DiT is pre-trained on IIT-CDIP document images, making it a strong backbone for Hebrew PDF classification.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

from src.data.dataset import HallucinationRiskDataset
from src.models.dit_classifier import DiTClassifier
from src.models.calibrator import TemperatureCalibrator
from src.utils.metrics import compute_metrics, compute_per_institution_metrics, compute_ece
from src.utils.visualization import (
    plot_confusion_matrix, plot_roc_curve, plot_pr_curve,
    plot_score_distribution
)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT           = Path('..').resolve()
METADATA_CSV   = ROOT / 'data' / 'metadata.csv'
RENDERED_DIR   = ROOT / 'data' / 'rendered'
SPLITS_DIR     = ROOT / 'data' / 'splits'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'dit'
BASELINE_CKPT  = ROOT / 'checkpoints' / 'baseline'
LOG_DIR        = ROOT / 'logs' / 'dit'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## 1 — Load Config


In [ ]:
with open(ROOT / 'configs' / 'dit.yaml') as f:
    cfg = yaml.safe_load(f)

print(yaml.dump(cfg, default_flow_style=False))

BATCH_SIZE    = cfg['training']['batch_size']
WD            = cfg['training']['weight_decay']
PATIENCE      = cfg['training']['early_stopping_patience']
STAGE1_EPOCHS = cfg['training']['stage1_epochs']
STAGE2_EPOCHS = cfg['training']['stage2_epochs']
STAGE3_EPOCHS = cfg['training']['stage3_epochs']
STAGE1_LR     = cfg['training']['stage1_lr']
STAGE2_LR     = cfg['training']['stage2_lr']
STAGE3_LR     = cfg['training']['stage3_lr']
MODEL_NAME    = cfg['model']['name']

## 2 — DiT Model Inspection


In [ ]:
print(f'Loading {MODEL_NAME} ...')
dit = DiTClassifier(model_name=MODEL_NAME, num_classes=1).to(device)

total_params     = sum(p.numel() for p in dit.parameters())
trainable_params = sum(p.numel() for p in dit.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

# ── Show layer structure ─────────────────────────────────────────────────────
n_encoder_layers = len(dit.backbone.encoder.layer)
hidden_size      = dit.backbone.config.hidden_size
print(f'\nBackbone encoder layers : {n_encoder_layers}')
print(f'Hidden size             : {hidden_size}')
print(f'Head input → output     : {hidden_size} → 1')

## 3 — Datasets and DataLoaders


In [ ]:
train_ds = HallucinationRiskDataset(METADATA_CSV, 'train', RENDERED_DIR, augment=True)
val_ds   = HallucinationRiskDataset(METADATA_CSV, 'val',   RENDERED_DIR, augment=False)
test_ds  = HallucinationRiskDataset(METADATA_CSV, 'test',  RENDERED_DIR, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## 4 — Training Utilities


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        n += len(labels)
    return total_loss / n


@torch.no_grad()
def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        logits = model(images).squeeze(1)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        n += len(labels)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    logits_np = np.concatenate(all_logits)
    labels_np = np.concatenate(all_labels)
    return total_loss / n, logits_np, labels_np


def run_stage(model, train_loader, val_loader, device, epochs, lr, wd, stage_name,
              early_stopping_patience=None):
    """Runs one training stage; returns (train_losses, val_losses, best_val_logits, best_val_labels)."""
    criterion = nn.BCEWithLogitsLoss()
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=wd)

    train_losses, val_losses, val_f1s = [], [], []
    best_val_f1   = -1.0
    best_state    = None
    no_improve    = 0

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_logits, val_labels = evaluate_epoch(model, val_loader, criterion, device)

        # Quick F1 at default 0.5 threshold for early-stopping signal
        from scipy.special import expit as sigmoid
        preds = (sigmoid(val_logits) > 0.5).astype(int)
        vf1   = f1_score(val_labels.astype(int), preds, pos_label=1, zero_division=0)

        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        val_f1s.append(vf1)

        print(f'[{stage_name}] Epoch {epoch}/{epochs}  train={tr_loss:.4f}  val={val_loss:.4f}  F1={vf1:.4f}')

        if vf1 > best_val_f1:
            best_val_f1   = vf1
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            best_logits   = val_logits.copy()
            best_labels   = val_labels.copy()
            no_improve    = 0
        else:
            no_improve += 1

        if early_stopping_patience and no_improve >= early_stopping_patience:
            print(f'  Early stopping triggered at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return train_losses, val_losses, val_f1s, best_logits, best_labels


def plot_stage_curves(train_losses, val_losses, val_f1s, stage_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, label='Train loss')
    ax1.plot(epochs, val_losses,   label='Val loss')
    ax1.set_title(f'{stage_name} — Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax2.plot(epochs, val_f1s, color='darkorange', label='Val F1')
    ax2.set_title(f'{stage_name} — Val F1')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    plt.tight_layout()
    plt.show()

## 5 — Stage 1: Head-Only Training

Backbone is fully frozen; only the linear classification head is trained.
This warms up the head to the document distribution before touching backbone weights.


In [ ]:
dit.freeze_backbone()
s1_trainable = sum(p.numel() for p in dit.parameters() if p.requires_grad)
print(f'Stage 1 trainable params: {s1_trainable:,}  (head only)')

s1_train_losses, s1_val_losses, s1_val_f1s, s1_logits, s1_labels = run_stage(
    dit, train_loader, val_loader, device,
    epochs=STAGE1_EPOCHS, lr=STAGE1_LR, wd=WD,
    stage_name='Stage1-HeadOnly'
)
plot_stage_curves(s1_train_losses, s1_val_losses, s1_val_f1s, 'Stage 1 — Head Only')

# Save stage 1 checkpoint
torch.save({'model_state_dict': dit.state_dict(), 'stage': 1}, CHECKPOINT_DIR / 'dit_stage1.pt')
print('Stage 1 checkpoint saved.')

## 6 — Stage 2: Top Blocks Unfrozen

Unfreezes the last 2 encoder blocks in addition to the head.
Lower learning rate to avoid catastrophic forgetting.


In [ ]:
dit.unfreeze_top_blocks(n_blocks=2)
s2_trainable = sum(p.numel() for p in dit.parameters() if p.requires_grad)
print(f'Stage 2 trainable params: {s2_trainable:,}  (head + top 2 encoder blocks)')

s2_train_losses, s2_val_losses, s2_val_f1s, s2_logits, s2_labels = run_stage(
    dit, train_loader, val_loader, device,
    epochs=STAGE2_EPOCHS, lr=STAGE2_LR, wd=WD,
    stage_name='Stage2-Top2Blocks'
)
plot_stage_curves(s2_train_losses, s2_val_losses, s2_val_f1s, 'Stage 2 — Top 2 Blocks')

torch.save({'model_state_dict': dit.state_dict(), 'stage': 2}, CHECKPOINT_DIR / 'dit_stage2.pt')
print('Stage 2 checkpoint saved.')

## 7 — Stage 3: Full Fine-Tune

All parameters unlocked. Very low learning rate with early stopping on val F1.


In [ ]:
dit.unfreeze_all()
s3_trainable = sum(p.numel() for p in dit.parameters() if p.requires_grad)
print(f'Stage 3 trainable params: {s3_trainable:,}  (full model)')

s3_train_losses, s3_val_losses, s3_val_f1s, s3_logits, s3_labels = run_stage(
    dit, train_loader, val_loader, device,
    epochs=STAGE3_EPOCHS, lr=STAGE3_LR, wd=WD,
    stage_name='Stage3-FullFineTune',
    early_stopping_patience=PATIENCE
)
plot_stage_curves(s3_train_losses, s3_val_losses, s3_val_f1s, 'Stage 3 — Full Fine-Tune')

# Best overall is stage-3 model (early stopping took care of overfitting)
torch.save({'model_state_dict': dit.state_dict(), 'stage': 3}, CHECKPOINT_DIR / 'dit_stage3.pt')
print('Stage 3 checkpoint saved.')

## 8 — DiT Calibration and Evaluation


In [ ]:
# Best val logits/labels come from stage 3
dit_val_logits = s3_logits
dit_val_labels = s3_labels

# ── Temperature calibration ──────────────────────────────────────────────────
dit_cal = TemperatureCalibrator()
dit_cal.calibrate(dit_val_logits, dit_val_labels)
print(f'DiT temperature: {dit_cal.temperature:.4f}')

dit_val_probs  = dit_cal.predict(dit_val_logits)
dit_thresholds = dit_cal.get_thresholds(
    dit_val_probs, dit_val_labels, target_false_safe_rate=0.05
)
print(f'T_low={dit_thresholds["T_low"]:.3f}  T_high={dit_thresholds["T_high"]:.3f}')

# ── Full metrics ─────────────────────────────────────────────────────────────
dit_metrics = compute_metrics(dit_val_labels, dit_val_probs, dit_thresholds)
print('\nDiT Val Metrics:')
for k, v in dit_metrics.items():
    print(f'  {k:25s}: {v:.4f}')

# ── Visualizations ───────────────────────────────────────────────────────────
plot_score_distribution(
    dit_val_probs, dit_val_labels.astype(int),
    output_path=str(LOG_DIR / 'dit_score_dist.png')
)
from IPython.display import Image as IPImage
IPImage(str(LOG_DIR / 'dit_score_dist.png'))

## 9 — DiT vs Baselines Comparison


In [ ]:
# Load baseline metrics saved by notebook 04
baseline_csv = BASELINE_CKPT / 'baseline_comparison.csv'
if baseline_csv.exists():
    baseline_df = pd.read_csv(baseline_csv, index_col='model')
else:
    print('Baseline comparison CSV not found. Run notebook 04 first.')
    baseline_df = pd.DataFrame()

dit_row = pd.DataFrame([dit_metrics], index=['DiT-Base'])
dit_row.index.name = 'model'
full_comparison = pd.concat([baseline_df[list(dit_metrics.keys())] if not baseline_df.empty else pd.DataFrame(), dit_row])

print('\nFull Model Comparison (Val Set):')
display_cols = ['f1', 'false_safe_rate', 'review_rate', 'roc_auc', 'pr_auc', 'ece']
print(full_comparison[display_cols].to_string(float_format='{:.4f}'.format))

# ── ROC curves on the same axes ───────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc
from scipy.special import expit as sigmoid

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')

for model_name, val_probs, val_labels in [
    ('DiT-Base', dit_val_probs, dit_val_labels),
]:
    if len(np.unique(val_labels)) >= 2:
        fpr, tpr, _ = roc_curve(val_labels.astype(int), val_probs)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f'{model_name} (AUC={roc_auc:.3f})')

# Attempt to overlay baselines if checkpoints are present
import pickle
for bname, bckpt_path, bcal_path in [
    ('ResNet50',  BASELINE_CKPT / 'resnet50_best.pt',      BASELINE_CKPT / 'resnet50_calibrator.pkl'),
    ('ViT-Base',  BASELINE_CKPT / 'vit_base_best.pt',      BASELINE_CKPT / 'vit_base_calibrator.pkl'),
]:
    if bckpt_path.exists() and bcal_path.exists():
        ckpt = torch.load(bckpt_path, map_location='cpu')
        # metrics stored in ckpt; we just draw a placeholder text if no probs saved
        roc_auc = ckpt.get('metrics', {}).get('roc_auc', float('nan'))
        if not np.isnan(roc_auc):
            ax.annotate(f'{bname} AUC={roc_auc:.3f}', xy=(0.6, 0.3 - 0.08 * ['ResNet50','ViT-Base'].index(bname)),
                        fontsize=9, color='gray')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models (Val Set)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 10 — Diagnostic Set: Borderline Cases

The **diagnostic set** consists of pages whose rubric `risk_score` falls in the range **4–6** (inclusive).
These are the hardest cases — partially filled forms, mixed handwriting/print, ambiguous content.
We evaluate separately to understand model behaviour on borderline pages.


In [ ]:
# Load metadata to get risk_score column
meta_df = pd.read_csv(METADATA_CSV)
print(f'Metadata rows: {len(meta_df)}')
print(meta_df.columns.tolist())

# Build a mapping from page_id to index in the val dataset
# (Assumes HallucinationRiskDataset preserves row order from metadata filtered to split='val')
val_meta = meta_df[meta_df['split'] == 'val'].reset_index(drop=True) if 'split' in meta_df.columns else meta_df.iloc[:len(val_ds)].reset_index(drop=True)

if 'risk_score' in val_meta.columns:
    borderline_mask = val_meta['risk_score'].between(4, 6)
    print(f'Borderline val pages (risk_score 4-6): {borderline_mask.sum()} / {len(val_meta)}')

    bl_probs  = dit_val_probs[borderline_mask.values]
    bl_labels = dit_val_labels[borderline_mask.values]

    if len(bl_probs) > 0 and len(np.unique(bl_labels)) >= 2:
        bl_metrics = compute_metrics(bl_labels, bl_probs, dit_thresholds)
        print('\nDiagnostic Set Metrics (risk_score 4–6):')
        for k, v in bl_metrics.items():
            print(f'  {k:25s}: {v:.4f}')
    else:
        print('Not enough borderline samples or single-class for full metrics.')
else:
    print('risk_score column not found in metadata — skipping diagnostic eval.')

## 11 — Save Best Checkpoint


In [ ]:
best_ckpt = {
    'model_state_dict': dit.state_dict(),
    'thresholds':  dit_thresholds,
    'metrics':     dit_metrics,
    'temperature': dit_cal.temperature,
    'stage':       3,
    'model_name':  MODEL_NAME,
}
torch.save(best_ckpt, CHECKPOINT_DIR / 'dit_best.pt')
dit_cal.save(str(CHECKPOINT_DIR / 'dit_calibrator.pkl'))

print(f'Best DiT checkpoint saved → {CHECKPOINT_DIR / "dit_best.pt"}')
print(f'Calibrator saved          → {CHECKPOINT_DIR / "dit_calibrator.pkl"}')

# ── Save full comparison for notebook 06 ─────────────────────────────────────
full_comparison.to_csv(CHECKPOINT_DIR / 'full_model_comparison.csv')
print('Full comparison CSV saved.')